# T10 #

a) Проверить гипотезу с согласии данных с законом равном. распределения через критерий Колмогорова и сравнить с проверкой через критерий $\chi^2$

In [32]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.style import Style
from rich.text import Text
from scipy.special import kolmogorov
import numpy as np
import math

alpha = 0.05
n = 100
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

def F_n(x, mean, std):
    return 0.5 * (1 + math.erf((x - mean) / (np.sqrt(2) * std)))

console = Console()
table = Table()

In [33]:

F_emp = np.array([sum(m[:i]) for i in range(len(m) + 1)]) / n
F_th = np.arange(10) / 10

# Выражение для ~delta из критерия
delta = np.sqrt(n) * np.max([max(np.abs(F_th[i] - F_emp[i]), np.abs(F_th[i] - F_emp[i + 1])) for i in range(F_th.size)])

# Расчет p-value = P(delta >= ~delta|H0) = 1 - Kolmogorov(~delta)
p_value = kolmogorov(delta)

print(f"p-value по критерию Колмогорова: {p_value}")

p-value по критерию Колмогорова: 0.039681879538114355


Берем $\alpha = 0.05$. По $\chi^2$ нет оснований отвергать гипотезу, но критерий Колмогорова (более мощный) отвергает.

b) Проверить гипотезу с согласии данных с законом норм. распределения через критерий Колмогорова и сравнить с проверкой через критерий $\chi^2$

Через $\chi^2$

In [34]:
# Полуинтервалы для групп событий для нормального распределения
intervals = np.array([(-np.inf, 1)] + [(i, i + 1) for i in range(1, 9)] + [(9, np.inf)])
sample = np.repeat(np.arange(len(m)), m)
std = np.sqrt(np.var(sample) * n / (n - 1))
alpha = np.mean(sample)

# Выборочная функция распр.
def F_n_sampled(x): 
    return F_n(x, alpha, std)

# Вероятности из норм. распр.
P = [F_n_sampled(i[1]) - F_n_sampled(i[0]) for i in intervals]
# Распределение хи-2
delta = sum((n * P[i] - m[i]) ** 2 / (n * P[i]) for i in range(10))
print(f"Delta по ОМПГ: {delta}")
print(f"p - value = 0.018")

Delta по ОМПГ: 16.871067048068763
p - value = 0.018


Отвергаем, p-value < 0.05

Используем парам. bootstrap (для Колмогорова)

In [35]:
bootstrap_iter = 50000
bootstrap_delta = []

x = np.arange(10)
delta_sampled = np.sqrt(n) * np.max([max(np.abs(F_n_sampled(x[i]) - F_emp[i]), np.abs(F_n_sampled(x[i]) - F_emp[i + 1])) for i in x])

for i in range(bootstrap_iter):
  random_sample = np.array(sorted(np.random.normal(alpha, std, n)))
  mu = random_sample.mean()
  sigma = np.sqrt(random_sample.var() * n / (n - 1))
  F_bootstrap_emp = [i / n for i in range(n + 1)]

  def F_bootstrap_sampled(j):
    return F_n(random_sample[j], mu, sigma)

  sup = np.max([max(np.abs(F_bootstrap_sampled(j) - F_bootstrap_emp[j]), np.abs(F_bootstrap_sampled(j) - F_bootstrap_emp[j + 1]))
  for j in range(len(random_sample))])
  bootstrap_delta.append(np.sqrt(n) * sup)

bootstrap_delta = np.array(bootstrap_delta)
p_value = len(bootstrap_delta[bootstrap_delta >= delta_sampled]) / bootstrap_iter

print(f"p - value по кр. Колмогорова: {p_value}")

p - value по кр. Колмогорова: 0.01532


Следовательно, основная гипотеза отвергается (p - value < 0.05), как и в случае с критерием $\chi^2$, даже с большей уверенностью